In [ ]:
"""
Tool eksternal: calculate_bmi
  - Fungsi sederhana untuk menghitung Body Mass Index (BMI) pasien.
  - Digunakan sebagai "tool call" agar agent bisa melakukan perhitungan nyata.
  - Input: berat (kg), tinggi (cm)
  - Output: nilai BMI
"""

# Tool eksternal: kalkulator BMI
def calculate_bmi(weight_kg, height_cm):
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    print(f"BMI pasien adalah: {bmi:.2f}")
    return bmi

In [ ]:
"""
Class AdvancedAIAgentDoctor
  - Merepresentasikan AI Agent versi lebih kompleks karena bisa memanggil tool eksternal.
  - Memiliki knowledge base awal berisi penyakit dan gejala terkait.
  - Menyimpan pengalaman (feedback dokter) untuk proses adaptasi.
"""

class AdvancedAIAgentDoctor:
    def __init__(self):
        # Knowledge base awal
        self.knowledge_base = {
            "diabetes": ["gula darah tinggi", "sering haus", "sering kencing"],
            "flu": ["demam", "batuk", "pilek"],
            "obesitas": ["bmi tinggi", "kelelahan", "nyeri sendi"]
        }
        self.experience = []  # menyimpan feedback dokter

    # 1. Perseption
    """
    Metode perceive()
    - Modul persepsi AI Agent.
    - Menerima data pasien (gejala, berat, tinggi).
    - Menampilkan data yang diterima agar bisa diproses lebih lanjut.
    """
    def perceive(self, patient_data):
      symptoms = patient_data.get("symptoms", [])
      weight = patient_data.get("weight", "tidak tersedia")
      height = patient_data.get("height", "tidak tersedia")
      print(f"Agent menerima data pasien dengan gejala: {', '.join(symptoms)}, berat badan: {weight} kg, tinggi badan: {height} cm")
      return patient_data

    # 2. Reasoning awal
    """
    Metode reason()
    - Modul penalaran awal (Reasoning).
    - Memeriksa gejala pasien dengan knowledge base.
    - Menentukan kemungkinan penyakit yang sesuai.
    - Output berupa list penyakit yang mungkin.
    """
    def reason(self, patient_data):
        patient_symptoms = patient_data.get("symptoms", [])
        possible_diseases = []
        for disease, symptoms in self.knowledge_base.items():
            if any(symptom in patient_symptoms for symptom in symptoms):
                possible_diseases.append(disease)
        if possible_diseases:
            print(f"Agent menilai kemungkinan penyakit: {', '.join(possible_diseases)}")
        else:
            print("Agent menilai: Tidak terdeteksi penyakit spesifik")
        return possible_diseases


    # 3. Planning
    """
    Metode plan()
    - Modul perencanaan (Planning).
    - Berdasarkan hasil reasoning, membuat langkah-langkah tindakan:
        • Diabetes → tes HbA1c tambahan
        • Flu → cek suhu & observasi gejala
        • Obesitas → hitung BMI pasien
    - Jika tidak ada penyakit yang terdeteksi → pemeriksaan umum.
    """
    def plan(self, possible_diseases):
        plan_steps = []
        for disease in possible_diseases:
            if disease == "diabetes":
                plan_steps.append("Lakukan tes HbA1c tambahan")
            elif disease == "flu":
                plan_steps.append("Periksa suhu dan observasi gejala")
            elif disease == "obesitas":
                plan_steps.append("Hitung BMI pasien")
        if not plan_steps:
            plan_steps.append("Lakukan pemeriksaan umum")
        print(f"Agent menyusun rencana tindakan: {', '.join(plan_steps)}")
        return plan_steps

    # 4. Action + Tool Calling
    """
    Metode act()
    - Modul tindakan (Action) + pemanggilan tool eksternal.
    - Mengeksekusi langkah-langkah yang sudah direncanakan.
    - Jika langkah memerlukan tool (contoh: hitung BMI), memanggil fungsi calculate_bmi.
    - Menghasilkan output akhir berupa diagnosis atau rekomendasi tindakan.
    """
    def act(self, plan_steps, patient_data):
        final_actions = []
        for step in plan_steps:
            if "BMI" in step:
                weight = patient_data.get("weight", 70)
                height = patient_data.get("height", 170)
                bmi = calculate_bmi(weight, height)
                if bmi >= 25:
                    final_actions.append("Diagnosis: pasien obesitas")
                else:
                    final_actions.append("Diagnosis: berat badan pasien normal")
            else:
                final_actions.append(step)
        print(f"Tindakan yang dilakukan Agent: {', '.join(final_actions)}")
        return final_actions

    # 5. Learning
    """
    Metode learn()
    - Modul pembelajaran (Learning).
    - Menyimpan feedback dari dokter untuk kasus tertentu.
    - Feedback bisa berupa "diagnosa benar" atau "salah diagnosa".
    """
    def learn(self, feedback):
        self.experience.append(feedback)
        print(f"Agent belajar dari feedback dokter: {feedback}")

    # 6. Adaptation
    """
    Metode adapt()
    - Modul adaptasi (Adaptation).
    - Berdasarkan feedback, agent bisa memperbarui knowledge base.
    - Contoh: jika diagnosa salah, menambahkan gejala baru ke database diabetes.
    - Dilengkapi dengan pengecekan agar tidak terjadi duplikasi gejala.
    """
    def adapt(self):
        if "salah diagnosa" in self.experience:
            if "kelelahan" not in self.knowledge_base["diabetes"]:
                self.knowledge_base["diabetes"].append("kelelahan")
                print("Agent menambahkan gejala baru 'kelelahan' ke database diabetes")
        else:
            print("Agent menilai: Sistem berjalan baik, tidak perlu adaptasi")

    # 7. Full simulation untuk 1 pasien
    """
    Metode run_case()
    - Menggabungkan seluruh alur kerja AI Agent untuk satu pasien.
    - Urutannya: perceive → reason → plan → act → learn → adapt.
    - Bisa menerima feedback opsional dari dokter.
    """
    def run_case(self, patient_data, feedback=None):
        self.perceive(patient_data)
        possible_diseases = self.reason(patient_data)
        plan_steps = self.plan(possible_diseases)
        self.act(plan_steps, patient_data)
        if feedback:
            self.learn(feedback)
            self.adapt()

In [ ]:
# Simulasi
"""
Simulasi
   - Membuat instance agent: agent = AdvancedAIAgentDoctor()
   - Menjalankan simulasi kasus pasien:
       • patient1: gejala, berat badan, tinggi badan
       • Output: rekomendasi tindakan atau diagnosa
   - Feedback digunakan untuk pembelajaran dan adaptasi knowledge base.
"""

agent = AdvancedAIAgentDoctor()

In [ ]:
# Kasus pasien 1
patient1 = {"symptoms": ["kelelahan", "sering kencing"], "weight": 90, "height": 165}
agent.run_case(patient1, feedback="diagnosa benar")

Agent menerima data pasien dengan gejala: kelelahan, sering kencing, berat badan: 90 kg, tinggi badan: 165 cm
Agent menilai kemungkinan penyakit: diabetes, obesitas
Agent menyusun rencana tindakan: Lakukan tes HbA1c tambahan, Hitung BMI pasien
BMI pasien adalah: 33.06
Tindakan yang dilakukan Agent: Lakukan tes HbA1c tambahan, Diagnosis: pasien obesitas
Agent belajar dari feedback dokter: diagnosa benar
Agent menilai: Sistem berjalan baik, tidak perlu adaptasi


In [ ]:
# Kasus pasien 2
patient2 = {"symptoms": ["kelelahan", "sering kencing"], "weight": 90, "height": 165}
agent.run_case(patient2, feedback="salah diagnosa")

Agent menerima data pasien dengan gejala: kelelahan, sering kencing, berat badan: 90 kg, tinggi badan: 165 cm
Agent menilai kemungkinan penyakit: diabetes, obesitas
Agent menyusun rencana tindakan: Lakukan tes HbA1c tambahan, Hitung BMI pasien
BMI pasien adalah: 33.06
Tindakan yang dilakukan Agent: Lakukan tes HbA1c tambahan, Diagnosis: pasien obesitas
Agent belajar dari feedback dokter: salah diagnosa
Agent menambahkan gejala baru 'kelelahan' ke database diabetes


In [ ]:
# Kasus pasien 3 (flu)
patient3 = {"symptoms": ["demam", "batuk"], "weight": 60, "height": 170}
agent.run_case(patient3, feedback="diagnosa benar")

Agent menerima data pasien dengan gejala: demam, batuk, berat badan: 60 kg, tinggi badan: 170 cm
Agent menilai kemungkinan penyakit: flu
Agent menyusun rencana tindakan: Periksa suhu dan observasi gejala
Tindakan yang dilakukan Agent: Periksa suhu dan observasi gejala
Agent belajar dari feedback dokter: diagnosa benar


In [ ]:
# Kasus pasien 4 (obesitas)
patient4 = {"symptoms": ["kelelahan", "nyeri sendi", "bmi tinggi"], "weight": 85, "height": 160}
agent.run_case(patient4)

Agent menerima data pasien dengan gejala: kelelahan, nyeri sendi, bmi tinggi, berat badan: 85 kg, tinggi badan: 160 cm
Agent menilai kemungkinan penyakit: diabetes, obesitas
Agent menyusun rencana tindakan: Lakukan tes HbA1c tambahan, Hitung BMI pasien
BMI pasien adalah: 33.20
Tindakan yang dilakukan Agent: Lakukan tes HbA1c tambahan, Diagnosis: pasien obesitas


In [ ]:
# Menampilkan knowledge base terbaru
print("📖 Knowledge base terbaru:", agent.knowledge_base)

📖 Knowledge base terbaru: {'diabetes': ['gula darah tinggi', 'sering haus', 'sering kencing', 'kelelahan'], 'flu': ['demam', 'batuk', 'pilek'], 'obesitas': ['bmi tinggi', 'kelelahan', 'nyeri sendi']}
